In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

In [ ]:
import numpy as np
import torch
import pandas as pd
import json
import shutil

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [ ]:
train_df = pd.read_csv('/content/sentiment_train_df.csv')
val_df = pd.read_csv('/content/sentiment_val_df.csv')

In [ ]:
model_name = "xlm-roberta-large"

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {v: k for k, v in label2id.items()}

train_data = train_df[["text_for_sentiment", "aspect", "label_for_sentiment"]].copy()
val_data   = val_df[["text_for_sentiment", "aspect", "label_for_sentiment"]].copy()

train_data["text_for_sentiment"] = train_data["text_for_sentiment"].astype(str)
val_data["text_for_sentiment"] = val_data["text_for_sentiment"].astype(str)

train_data["aspect"] = train_data["aspect"].astype(str)
val_data["aspect"] = val_data["aspect"].astype(str)

train_data["label"] = train_data["label_for_sentiment"].map(label2id)
val_data["label"] = val_data["label_for_sentiment"].map(label2id)

train_data = train_data.dropna(subset=["label"])
val_data = val_data.dropna(subset=["label"])

train_data["label"] = train_data["label"].astype(int)
val_data["label"] = val_data["label"].astype(int)

In [ ]:
train_data["input_text"] = (
    train_data["text_for_sentiment"] +
    " </s> aspect: " +
    train_data["aspect"]
)

val_data["input_text"] = (
    val_data["text_for_sentiment"] +
    " </s> aspect: " +
    val_data["aspect"]
)

In [ ]:
train_dataset = Dataset.from_pandas(train_data[["input_text", "label"]])
val_dataset = Dataset.from_pandas(val_data[["input_text", "label"]])

In [ ]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_text', 'label'],
    num_rows: 3333
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 840
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_sentiment_data(example):
    encoding = tokenizer(
        example["input_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    encoding["labels"] = example["label"]
    return encoding

train_dataset = train_dataset.map(tokenize_sentiment_data)
val_dataset = val_dataset.map(tokenize_sentiment_data)

cols_to_keep = ["input_ids", "attention_mask", "labels"]

train_dataset.set_format(type="torch", columns=cols_to_keep)
val_dataset.set_format(type="torch", columns=cols_to_keep)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3333 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./sentiment_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    num_train_epochs=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),

    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=7,
            early_stopping_threshold=0.0
        )
    ]
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Micro F1,Macro Precision,Macro Recall
1,0.836920,0.790475,0.607143,0.356753,0.607143,0.474753,0.397704
2,0.395507,0.358061,0.889286,0.678299,0.889286,0.924358,0.659979
3,0.288531,0.298826,0.900000,0.762330,0.900000,0.931958,0.718729
4,0.260463,0.378052,0.896429,0.759648,0.896429,0.934250,0.713572
5,0.273167,0.292188,0.904762,0.766053,0.904762,0.935467,0.722645
6,0.291571,0.303727,0.904762,0.766956,0.904762,0.878783,0.727145
7,0.185800,0.319036,0.922619,0.813959,0.922619,0.879227,0.779055
8,0.196966,0.420999,0.923810,0.790432,0.923810,0.856639,0.759965
9,0.163615,0.377959,0.938095,0.833421,0.938095,0.925239,0.791918
10,0.135679,0.454207,0.927381,0.797513,0.927381,0.885365,0.759685


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import json

for ckpt in ["./sentiment_model/checkpoint-3762", "./sentiment_model/checkpoint-4598"]:
    with open(f"{ckpt}/trainer_state.json") as f:
        state = json.load(f)
    print(ckpt, state["epoch"], state.get("best_metric"))

./sentiment_model/checkpoint-3762 18.0 0.8487955254608512
./sentiment_model/checkpoint-4598 22.0 0.8487955254608512


In [ ]:
model.save_pretrained("./sentiment_epoch18_best")
tokenizer.save_pretrained("./sentiment_epoch18_best")

with open("./sentiment_epoch18_best/label2id.json", "w") as f:
    json.dump(label2id, f)

with open("./sentiment_epoch18_best/id2label.json", "w") as f:
    json.dump(id2label, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# shutil.make_archive(
#     "/kaggle/working/best_sentiment_model",
#     "zip",
#     "./best_sentiment_model"
# )

In [ ]:
import shutil

shutil.make_archive("sentiment_epoch18_best", 'zip', "./sentiment_epoch18_best")

'/content/sentiment_epoch18_best.zip'

In [ ]:
import shutil

shutil.copytree(
    "./sentiment_epoch18_best",
    "/content/drive/MyDrive/Colab Notebooks/sentiment_epoch18_best"
)

'/content/drive/MyDrive/Colab Notebooks/sentiment_epoch18_best'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json

model_path = "/content/sentiment_epoch18_best"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=1024, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/sentiment_epoch18_best"

def predict_sentiment(text, aspect):
    input_text = str(text) + " </s> aspect: " + str(aspect)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))

    return {
        "label": model.config.id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "probs": {
            model.config.id2label[i]: float(probs[i])
            for i in range(len(probs))
        }
    }

In [ ]:
predict_sentiment(
    "لا يوجد الدفع بالبطاقة عند الاستلام",
    "app_experience"
)

{'label': 'negative',
 'confidence': 0.9999773502349854,
 'probs': {'negative': 0.9999773502349854,
  'neutral': 1.1698373782564886e-05,
  'positive': 1.097647054848494e-05}}